# Graph Feature Engineering

This notebook extracts graph-based features from the logistics network.

Objectives:

- Calculate node-level graph metrics
- Generate ML-ready graph features
- Export graph_features.csv
- Prepare data for graph-enhanced ETA prediction

In [1]:
import pandas as pd
import networkx as nx

## Load Corridor Dataset

In [2]:
corridors = pd.read_csv(
    "../data/processed/corridor_stats.csv"
)

corridors.head()

,source_center,destination_center,avg_actual_time,avg_osrm_time,trip_count,delay_ratio
0,IND000000AAL,IND411033AAA,63.702703,22.216216,37,2.867397
1,IND000000AAS,IND783370AAC,50.555556,25.888889,18,1.952790
2,IND000000ABA,IND683565AAA,30.846154,19.307692,13,1.597610
3,IND000000ABD,IND562132AAA,336.868056,201.986111,144,1.667778
4,IND000000ABG,IND501359AAF,111.170732,20.707317,41,5.368669


## Rebuild Logistics Graph

In [3]:
G = nx.DiGraph()

for _, row in corridors.iterrows():

    G.add_edge(
        row["source_center"],
        row["destination_center"],
        weight=row["delay_ratio"]
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1483
Edges: 2203


## Degree Centrality

In [4]:
degree = nx.degree_centrality(G)

## Betweenness Centrality

In [5]:
betweenness = nx.betweenness_centrality(G)

## PageRank (based on):
- number of connections
- quality of connections

In [6]:
pagerank = nx.pagerank(
    G,
    weight="weight"
)

## Clustering Coefficient

In [7]:
clustering = nx.clustering(
    G.to_undirected()
)

## Create Feature Table

In [8]:
graph_features = pd.DataFrame({
    "hub": list(G.nodes())
})

graph_features.head()

,hub
0,IND000000AAL
1,IND411033AAA
2,IND000000AAS
3,IND783370AAC
4,IND000000ABA


In [9]:
graph_features["degree_centrality"] = (
    graph_features["hub"]
    .map(degree)
)

graph_features["betweenness"] = (
    graph_features["hub"]
    .map(betweenness)
)

graph_features["pagerank"] = (
    graph_features["hub"]
    .map(pagerank)
)

graph_features["clustering"] = (
    graph_features["hub"]
    .map(clustering)
)

graph_features.head()

,hub,degree_centrality,betweenness,pagerank,clustering
0,IND000000AAL,0.00135,0.000000,0.000585,0.00000
1,IND411033AAA,0.02834,0.038914,0.007016,0.05665
2,IND000000AAS,0.00135,0.000004,0.000377,0.00000
3,IND783370AAC,0.00135,0.000004,0.000467,0.00000
4,IND000000ABA,0.00135,0.000419,0.000414,1.00000


In [10]:
graph_features.shape

(1483, 5)

In [11]:
graph_features.sort_values(
    by="pagerank",
    ascending=False
).head(10)

,hub,degree_centrality,betweenness,pagerank,clustering
20,IND000000ACB,0.062078,0.189313,0.012211,0.036821
58,IND501359AAE,0.037787,0.080381,0.011314,0.043590
7,IND562132AAA,0.047908,0.110931,0.010388,0.050170
19,IND160002AAC,0.037112,0.045147,0.009243,0.034146
1,IND411033AAA,0.028340,0.038914,0.007016,0.056650
61,IND712311AAA,0.027665,0.062931,0.006954,0.044335
42,IND131028AAB,0.024291,0.044809,0.006859,0.039683
55,IND421302AAG,0.036437,0.054761,0.006776,0.067227
88,IND110037AAM,0.028340,0.031620,0.006124,0.071429
47,IND209304AAA,0.019568,0.033524,0.006022,0.034632


### Observation

PageRank identifies the most influential hubs in the logistics network.

These hubs play a critical role in shipment movement and ETA performance.

In [12]:
graph_features.to_csv(
    "../data/processed/graph_features.csv",
    index=False
)

print("graph_features.csv saved")

graph_features.csv saved


In [13]:
saved = pd.read_csv(
    "../data/processed/graph_features.csv"
)

saved.head()

,hub,degree_centrality,betweenness,pagerank,clustering
0,IND000000AAL,0.00135,0.000000,0.000585,0.00000
1,IND411033AAA,0.02834,0.038914,0.007016,0.05665
2,IND000000AAS,0.00135,0.000004,0.000377,0.00000
3,IND783370AAC,0.00135,0.000004,0.000467,0.00000
4,IND000000ABA,0.00135,0.000419,0.000414,1.00000


# The graph_features.csv file contains:

- degree centrality
- betweenness centrality
- pagerank
- clustering coefficient

These features can now be merged with trip-level data for graph-enhanced ETA prediction.